# lab8_3_prompts

从上到下依次运行每个单元格（Shift + Enter）。标有“← 改这里”的地方是“改一改”练习要修改的位置。

### 第 1 步：加载模型

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",      # Stable Diffusion 1.5，生成 512×512 图片
    torch_dtype=torch.float16).to("cuda")                # 半精度，省显存

### 第 2 步：同一个路口，换四种天气

In [ ]:
base = "a city street intersection seen from a car dashboard camera, cars and traffic lights, realistic photo"
negative = "cartoon, painting, blurry, distorted, low quality, text, watermark"   # 反向提示词：不要什么
weathers = {"sunny": "sunny day, clear sky", "rain": "heavy rain, wet road, reflections",
            "night": "at night, street lights, headlights", "fog": "dense fog, low visibility"}
for name, w in weathers.items():
    g = torch.Generator("cuda").manual_seed(42)           # 固定随机种子，只让天气不同  ← 改这里
    img = pipe(f"{base}, {w}", negative_prompt=negative,
               num_inference_steps=25, guidance_scale=7.5, generator=g).images[0]
    img.save(f"lab8_3_{name}.png")
    print("已生成：", name)

### 第 3 步：改步数和引导强度，看看有什么变化

In [ ]:
prompt = f"{base}, heavy rain, wet road, reflections"
for steps in [5, 15, 30]:                                 # 去噪步数  ← 改这里
    g = torch.Generator("cuda").manual_seed(42)
    pipe(prompt, negative_prompt=negative, num_inference_steps=steps,
         guidance_scale=7.5, generator=g).images[0].save(f"lab8_3_steps{steps}.png")
for cfg in [1.0, 4.0, 12.0]:                              # 引导强度：多听提示词的话  ← 改这里
    g = torch.Generator("cuda").manual_seed(42)
    pipe(prompt, negative_prompt=negative, num_inference_steps=25,
         guidance_scale=cfg, generator=g).images[0].save(f"lab8_3_cfg{cfg:g}.png")

### 第 4 步：换随机种子——同样的提示词，每次画得不一样

In [ ]:
for seed in [1, 2, 3]:
    g = torch.Generator("cuda").manual_seed(seed)
    pipe(prompt, negative_prompt=negative, num_inference_steps=25,
         guidance_scale=7.5, generator=g).images[0].save(f"lab8_3_seed{seed}.png")
print("全部完成，请把图片填进实验记录表")